# 2026-10-18_2 scTenifoldKnk ONECUT1 / MAFK KO 显著 DR 基因计数图

基于完整 gene-level diffRegulation 结果表（非 target-pair summary），统计各 KO 网络中显著扰动基因数（DR genes，p.adj < 0.05，**排除被敲除 TF 自身**，沿用既定规则），按预测方向（direction：down / up）堆叠展示：

1. **pooled 版本**（每 TF 一张）：X = Central / Peripheral，Y = 显著 DR gene counts，Down/Up 堆叠，段内标注 counts
2. **stage 版本**（每 TF 一张）：按 region 分面（Central | Peripheral），分面内 X = Young / Old，Down/Up 堆叠，段内标注 counts

数据来源：ONECUT1 pooled = 2026-10-18 轮（0.01 放宽）、stage = 2026-10-15 轮（0.01）；MAFK pooled = 2026-10-18 轮（0.05）、stage = 2026-10-16 轮（0.05）。注意 stage 版本两分面 y 轴刻度独立（free_y），便于读取小计数（图中已注明）。

输出：`Plot/2026-10-18_{ONECUT1,MAFK}_KO_sig_gene_counts_by_region[_stage].pdf`（仅 PDF，白底简洁论文风格）。

In [ ]:
# ---- 1. 配置：路径 / 文件 / 颜色（集中修改处） --------------------------------
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
})
set.seed(1)

KNK_DIR  <- "/project/work/2026-10-13_scTenifoldKnk/Data/"
PLOT_DIR <- "/project/work/2026-10-13_scTenifoldKnk/Plot/"
DATE     <- "2026-10-18"
PADJ_CUTOFF <- 0.05

# 颜色（沿用 /project/work/color.ipynb palette3 中间调；浅色填充保证段内黑色标注可读）
cols.direction <- c(down = "#8491B4", up = "#F39B7F")
dir_labels   <- c(down = "Predicted down", up = "Predicted up")
region_labels <- c(central = "Central", peripheral = "Peripheral")
stage_labels  <- c(young = "Young", old = "Old")

## 输入文件清单（完整 gene-level diffRegulation）
FILES <- rbindlist(lapply(c("ONECUT1", "MAFK"), function(tf) {
  stgdate <- if (tf == "ONECUT1") "2026-10-15" else "2026-10-16"
  rbind(
    data.table(TF = tf, region = c("central", "peripheral"), stage = "pooled",
               file = paste0("2026-10-18_MG_", c("central", "peripheral"), "_", tf, "_diffRegulation.tsv")),
    data.table(TF = tf, region = rep(c("central", "peripheral"), each = 2),
               stage = rep(c("young", "old"), 2),
               file = paste0(stgdate, "_MG_", rep(c("central", "peripheral"), each = 2), "_",
                             rep(c("young", "old"), 2), "_", tf, "_diffRegulation.tsv"))
  )
}))
stopifnot(all(file.exists(file.path(KNK_DIR, FILES$file))))
print(FILES)

## 显著 DR 基因计数（p.adj < 0.05，排除被敲除 TF 自身）
cnt <- rbindlist(lapply(seq_len(nrow(FILES)), function(i) {
  d <- fread(file.path(KNK_DIR, FILES$file[i]))
  miss <- setdiff(c("gene", "p.adj", "direction"), names(d))
  if (length(miss) > 0) stop(FILES$file[i], " 缺少必需列: ", paste(miss, collapse = ", "))
  dd <- d[gene != FILES$TF[i] & p.adj < PADJ_CUTOFF]
  data.table(TF = FILES$TF[i], region = FILES$region[i], stage = FILES$stage[i],
             direction = c("down", "up"),
             n = c(sum(dd$direction == "down"), sum(dd$direction == "up")),
             total_sig = nrow(dd))
}))
cat("\n=== 显著 DR 基因计数（down/up）===\n")
print(cnt)

In [ ]:
# ---- 2. pooled 版本：显著 DR gene 计数（region x direction 堆叠）----------------
for (tf in c("ONECUT1", "MAFK")) {
  d <- cnt[TF == tf & stage == "pooled"]
  d[, Region := factor(region_labels[region], levels = unname(region_labels))]
  d[, direction := factor(direction, levels = c("down", "up"))]
  p <- ggplot(d, aes(Region, n, fill = direction)) +
    geom_col(width = 0.62, color = "grey25", linewidth = 0.25) +
    geom_text(aes(label = ifelse(n > 0, as.character(n), "")),
              position = position_stack(vjust = 0.5), size = 3.1, color = "black") +
    scale_fill_manual(values = cols.direction, labels = dir_labels, name = NULL) +
    scale_y_continuous(expand = expansion(mult = c(0.02, 0.08))) +
    labs(title = sprintf("%s virtual knockout (MG, pooled)", tf),
         x = NULL, y = "Number of significant DR genes") +
    theme_bw(base_size = 11) +
    theme(legend.position = "top", panel.grid.minor = element_blank())
  print(p)
  ggsave(file.path(PLOT_DIR, paste0(DATE, "_", tf, "_KO_sig_gene_counts_by_region.pdf")),
         p, width = 3.8, height = 3.6)
  cat("已保存:", DATE, "_", tf, "_KO_sig_gene_counts_by_region.pdf\n")
}

In [ ]:
# ---- 3. stage 版本：显著 DR gene 计数（region 分面 x stage x direction 堆叠）----
## free_y：Central / Peripheral 两分面 y 轴刻度独立（计数量级差异大，保证小计数段内标注可读）
for (tf in c("ONECUT1", "MAFK")) {
  d <- cnt[TF == tf & stage != "pooled"]
  d[, Region := factor(region_labels[region], levels = unname(region_labels))]
  d[, Stage := factor(stage_labels[stage], levels = unname(stage_labels))]
  d[, direction := factor(direction, levels = c("down", "up"))]
  p <- ggplot(d, aes(Stage, n, fill = direction)) +
    geom_col(width = 0.62, color = "grey25", linewidth = 0.25) +
    geom_text(aes(label = ifelse(n > 0, as.character(n), "")),
              position = position_stack(vjust = 0.5), size = 3.1, color = "black") +
    facet_grid(. ~ Region, scales = "free_y") +
    scale_fill_manual(values = cols.direction, labels = dir_labels, name = NULL) +
    scale_y_continuous(expand = expansion(mult = c(0.02, 0.08))) +
    labs(title = sprintf("%s virtual knockout (MG, by stage)", tf),
         x = NULL, y = "Number of significant DR genes",
         caption = "Note: y-axis scales differ between panels") +
    theme_bw(base_size = 11) +
    theme(legend.position = "top", panel.grid.minor = element_blank(),
          plot.caption = element_text(size = 7, hjust = 1))
  print(p)
  ggsave(file.path(PLOT_DIR, paste0(DATE, "_", tf, "_KO_sig_gene_counts_by_region_stage.pdf")),
         p, width = 6.2, height = 3.6)
  cat("已保存:", DATE, "_", tf, "_KO_sig_gene_counts_by_region_stage.pdf\n")
}
cat("\n输出文件:\n")
print(list.files(PLOT_DIR, pattern = paste0("^", DATE, ".*_KO_sig_gene_counts.*[.]pdf$"), full.names = TRUE))